# 🎵 Spotify Rolling Stones — Song Cohort Discovery
### Multi-Algorithm Unsupervised Learning Pipeline
**Dataset:** Rolling Stones discography on Spotify (1,610 tracks)  
**Objective:** Discover natural song cohorts using K-Means, Hierarchical Clustering, DBSCAN, and GMM; validate with supervised feature importance.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'figure.dpi':       120
})
PALETTE = ['#e94560','#0f3460','#533483','#00b4d8','#f4a261']
print('✅ Libraries loaded successfully.')

## 2. Load & Inspect Dataset

In [ ]:
df_raw = pd.read_csv('rolling_stones_spotify.csv')
print(f'Shape: {df_raw.shape}')
print(f'\nColumns: {list(df_raw.columns)}')
print(f'\nData Types:\n{df_raw.dtypes}')
print(f'\nNull Values:\n{df_raw.isnull().sum()}')
print(f'\nDuplicate Rows: {df_raw.duplicated().sum()}')
df_raw.head(3)

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Remove duplicates and nulls
df = df.drop_duplicates().dropna()

# Parse dates, extract year and decade
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year
df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'

# Convert duration to minutes for readability
df['duration_min'] = df['duration_ms'] / 60000

# Drop non-numeric identifier columns
drop_cols = ['name', 'album', 'id', 'uri', 'release_date', 'duration_ms']
df.drop(columns=drop_cols, inplace=True, errors='ignore')

# Audio feature columns used for clustering
AUDIO_FEATURES = ['acousticness','danceability','energy','instrumentalness',
                  'liveness','loudness','speechiness','tempo','valence','duration_min']

print(f'Clean dataset shape: {df.shape}')
df[AUDIO_FEATURES].describe().round(3)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- Popularity Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Popularity Distribution', fontsize=16, fontweight='bold', color='white')

sns.histplot(df['popularity'], bins=30, kde=True, color='#e94560', ax=axes[0])
axes[0].set_title('Histogram + KDE', color='white')
axes[0].set_xlabel('Popularity Score')

sns.boxplot(y=df['popularity'], color='#00b4d8', ax=axes[1])
axes[1].set_title('Box Plot', color='white')

plt.tight_layout()
plt.savefig('plot_01_popularity_distribution.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(f'Median popularity: {df["popularity"].median()}')

In [ ]:
# --- Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[AUDIO_FEATURES + ['popularity']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', annot=True, fmt='.2f',
            linewidths=0.4, annot_kws={'size':9}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_02_correlation_heatmap.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# --- Songs per Decade ---
decade_counts = df['decade'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(decade_counts.index, decade_counts.values,
              color=PALETTE[:len(decade_counts)], edgecolor='white', linewidth=0.5)
ax.bar_label(bars, color='white', fontsize=10)
ax.set_title('Tracks Released Per Decade', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Track Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot_03_tracks_per_decade.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# --- Feature Distributions (violin plots) ---
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Audio Feature Distributions', fontsize=16, fontweight='bold')
axes = axes.flatten()
for i, feat in enumerate(AUDIO_FEATURES):
    sns.violinplot(y=df[feat], ax=axes[i], color=PALETTE[i % len(PALETTE)], inner='box')
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel('')
plt.tight_layout()
plt.savefig('plot_04_feature_distributions.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 5. Feature Scaling

In [ ]:
X = df[AUDIO_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Mean (post-scaling): {X_scaled.mean(axis=0).round(4)}')
print(f'Std  (post-scaling): {X_scaled.std(axis=0).round(4)}')

## 6. Optimal K Selection — Elbow Method + Silhouette Analysis

In [ ]:
K_range = range(2, 12)
wcss, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal K Selection', fontsize=16, fontweight='bold')

# Elbow
axes[0].plot(list(K_range), wcss, 'o-', color='#e94560', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method (WCSS)', fontsize=13)
axes[0].set_xlabel('Number of Clusters K')
axes[0].set_ylabel('Within-Cluster Sum of Squares')
axes[0].grid(alpha=0.3)

# Silhouette
best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].plot(list(K_range), sil_scores, 's-', color='#00b4d8', linewidth=2, markersize=7)
axes[1].axvline(best_k, color='#f4a261', linestyle='--', label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score vs K', fontsize=13)
axes[1].set_xlabel('Number of Clusters K')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_05_optimal_k.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(f'\n>>> Best K by Silhouette Score: {best_k} (score={max(sil_scores):.4f})')
K_OPTIMAL = best_k

## 7. Silhouette Plot for Optimal K

In [ ]:
km_opt = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
km_labels = km_opt.fit_predict(X_scaled)

sil_vals = silhouette_samples(X_scaled, km_labels)
fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
colors = cm.tab10(np.linspace(0, 1, K_OPTIMAL))

for i in range(K_OPTIMAL):
    cluster_sil = np.sort(sil_vals[km_labels == i])
    size = cluster_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=colors[i], edgecolor=colors[i], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * size, f'C{i}')
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(X_scaled, km_labels), color='red', linestyle='--', label='Avg Score')
ax.set_title(f'Silhouette Plot — K={K_OPTIMAL}', fontsize=14, fontweight='bold')
ax.set_xlabel('Silhouette Coefficient')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.savefig('plot_06_silhouette_plot.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
df['cluster_kmeans'] = km_labels
print(f'K-Means cluster sizes:\n{pd.Series(km_labels).value_counts().sort_index()}')

## 8. Hierarchical (Agglomerative) Clustering + Dendrogram

In [ ]:
# Sample for dendrogram clarity
sample_idx = np.random.choice(len(X_scaled), size=min(200, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(18, 6))
dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=6,
           color_threshold=0.7 * max(Z[:, 2]),
           above_threshold_color='#888')
ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage, 200-song sample)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('plot_07_dendrogram.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# Full agglomerative clustering
agg = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage='ward')
df['cluster_hierarchical'] = agg.fit_predict(X_scaled)
print(f'Hierarchical cluster sizes:\n{pd.Series(df["cluster_hierarchical"]).value_counts().sort_index()}')
sil_agg = silhouette_score(X_scaled, df['cluster_hierarchical'])
print(f'Agglomerative Silhouette Score: {sil_agg:.4f}')

## 9. DBSCAN — Density-Based Clustering + Anomaly Detection

In [ ]:
# Tune eps using k-distance graph
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances = np.sort(distances[:, 4])[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_distances, color='#e94560', linewidth=1.5)
ax.set_title('k-Distance Graph (k=5) — Elbow indicates optimal eps', fontsize=13, fontweight='bold')
ax.set_xlabel('Points sorted by distance')
ax.set_ylabel('5th Nearest Neighbour Distance')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot_08_kdistance.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# Run DBSCAN
dbscan = DBSCAN(eps=1.5, min_samples=5)
df['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_noise = (df['cluster_dbscan'] == -1).sum()
print(f'DBSCAN → Clusters found: {n_clusters_db} | Outlier/noise songs: {n_noise}')
print(f'\nCluster distribution:\n{df["cluster_dbscan"].value_counts().sort_index()}')

In [ ]:
# Identify anomalous songs
anomaly_rows = df_raw.loc[df.index[df['cluster_dbscan'] == -1],
                          ['name','album','popularity','energy','acousticness','tempo']]
print(f'>>> {len(anomaly_rows)} Anomalous / Outlier Tracks Detected:')
anomaly_rows.head(15)

## 10. Gaussian Mixture Model (GMM) — Soft Probabilistic Clustering

In [ ]:
# BIC/AIC to select number of components
bic_scores, aic_scores = [], []
K_gmm_range = range(2, 10)

for k in K_gmm_range:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    gmm.fit(X_scaled)
    bic_scores.append(gmm.bic(X_scaled))
    aic_scores.append(gmm.aic(X_scaled))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(K_gmm_range), bic_scores, 'o-', color='#e94560', label='BIC', linewidth=2)
ax.plot(list(K_gmm_range), aic_scores, 's-', color='#00b4d8', label='AIC', linewidth=2)
ax.set_title('GMM Model Selection: BIC & AIC vs Components', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Score (lower = better)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot_09_gmm_bic_aic.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

best_k_gmm = list(K_gmm_range)[np.argmin(bic_scores)]
gmm_opt = GaussianMixture(n_components=best_k_gmm, covariance_type='full', random_state=42)
gmm_opt.fit(X_scaled)
df['cluster_gmm'] = gmm_opt.predict(X_scaled)
gmm_probs = gmm_opt.predict_proba(X_scaled)

print(f'Optimal GMM components (by BIC): {best_k_gmm}')
print(f'GMM cluster sizes:\n{pd.Series(df["cluster_gmm"]).value_counts().sort_index()}')
print(f'GMM Silhouette Score: {silhouette_score(X_scaled, df["cluster_gmm"]):.4f}')

## 11. Algorithm Comparison Summary

In [ ]:
try:
    sil_km   = silhouette_score(X_scaled, df['cluster_kmeans'])
    sil_agg2 = silhouette_score(X_scaled, df['cluster_hierarchical'])
    sil_gmm  = silhouette_score(X_scaled, df['cluster_gmm'])
    # DBSCAN: exclude noise points
    valid_db = df['cluster_dbscan'] != -1
    sil_db = silhouette_score(X_scaled[valid_db], df['cluster_dbscan'][valid_db]) if valid_db.sum() > 1 else 0
except:
    sil_km = sil_agg2 = sil_gmm = sil_db = 0

comparison = pd.DataFrame({
    'Algorithm': ['K-Means', 'Agglomerative', 'DBSCAN', 'GMM'],
    'Silhouette Score': [round(sil_km, 4), round(sil_agg2, 4), round(sil_db, 4), round(sil_gmm, 4)],
    'Clusters Found': [K_OPTIMAL, K_OPTIMAL, n_clusters_db, best_k_gmm],
    'Handles Noise': ['No', 'No', 'Yes', 'Partial'],
    'Soft Assignment': ['No', 'No', 'No', 'Yes']
})

print('=' * 65)
print('         CLUSTERING ALGORITHM COMPARISON SUMMARY')
print('=' * 65)
print(comparison.to_string(index=False))
print('=' * 65)
print(f'\n>>> Best algorithm by Silhouette: {comparison.loc[comparison["Silhouette Score"].idxmax(), "Algorithm"]}')

## 12. Dimensionality Reduction: PCA (2D + 3D)

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)

print(f'PCA 2D — Explained Variance: {pca2.explained_variance_ratio_.sum()*100:.1f}%')
print(f'PCA 3D — Explained Variance: {pca3.explained_variance_ratio_.sum()*100:.1f}%')

# 2D PCA plot
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=df['cluster_kmeans'],
                     cmap='tab10', s=15, alpha=0.7)
ax.set_title(f'PCA 2D — K-Means Clusters (K={K_OPTIMAL})', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('plot_10_pca2d.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# 3D PCA
fig = plt.figure(figsize=(12, 8))
ax3 = fig.add_subplot(111, projection='3d')
sc = ax3.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2],
                 c=df['cluster_kmeans'], cmap='tab10', s=12, alpha=0.7)
ax3.set_xlabel('PC1'); ax3.set_ylabel('PC2'); ax3.set_zlabel('PC3')
ax3.set_title(f'PCA 3D — K-Means Clusters', fontsize=14, fontweight='bold')
fig.colorbar(sc, ax=ax3, shrink=0.5, label='Cluster')
plt.tight_layout()
plt.savefig('plot_11_pca3d.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 13. t-SNE Visualization

In [ ]:
print('Running t-SNE (this may take ~30 seconds)...')
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('t-SNE 2D Embedding', fontsize=16, fontweight='bold')

# Colored by K-Means cluster
sc1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=df['cluster_kmeans'],
                      cmap='tab10', s=12, alpha=0.8)
axes[0].set_title('Colored by K-Means Cluster')
axes[0].set_xlabel('t-SNE Dim 1'); axes[0].set_ylabel('t-SNE Dim 2')
fig.colorbar(sc1, ax=axes[0], label='Cluster')

# Colored by popularity
sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=df['popularity'],
                      cmap='RdYlGn', s=12, alpha=0.8)
axes[1].set_title('Colored by Popularity Score')
axes[1].set_xlabel('t-SNE Dim 1'); axes[1].set_ylabel('t-SNE Dim 2')
fig.colorbar(sc2, ax=axes[1], label='Popularity')

plt.tight_layout()
plt.savefig('plot_12_tsne.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 14. Cluster Profiling — Musical Interpretation

In [ ]:
# Compute cluster mean profiles
profile = df.groupby('cluster_kmeans')[AUDIO_FEATURES + ['popularity']].mean().round(3)
print('Cluster Mean Feature Profiles:')
print(profile.to_string())

# Auto-label clusters based on dominant traits
def label_cluster(row):
    if row['energy'] > 0.75 and row['acousticness'] < 0.3:
        return 'High-Energy Rock'
    elif row['acousticness'] > 0.5 and row['energy'] < 0.5:
        return 'Acoustic & Mellow'
    elif row['danceability'] > 0.6 and row['valence'] > 0.5:
        return 'Groovy & Upbeat'
    elif row['instrumentalness'] > 0.3:
        return 'Instrumental / Experimental'
    elif row['liveness'] > 0.5:
        return 'Live Performance'
    else:
        return 'Mid-Tempo Classic'

cluster_labels = {i: label_cluster(profile.loc[i]) for i in profile.index}
df['cluster_name'] = df['cluster_kmeans'].map(cluster_labels)

print('\n>>> Cluster Musical Labels:')
for k, v in cluster_labels.items():
    count = (df['cluster_kmeans'] == k).sum()
    print(f'  Cluster {k}: "{v}" — {count} songs')

In [ ]:
# Heatmap of cluster profiles
profile_norm = (profile[AUDIO_FEATURES] - profile[AUDIO_FEATURES].min()) / \
               (profile[AUDIO_FEATURES].max() - profile[AUDIO_FEATURES].min())
profile_norm.index = [f'C{i}: {cluster_labels[i]}' for i in profile_norm.index]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_norm, cmap='RdYlGn', annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Cluster Feature Profiles (Normalized)', fontsize=14, fontweight='bold')
ax.set_xlabel('Audio Feature')
ax.set_ylabel('Cluster')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plot_13_cluster_profiles.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 15. Radar / Spider Charts — Audio Fingerprint per Cluster

In [ ]:
radar_features = ['acousticness','danceability','energy','instrumentalness',
                  'liveness','speechiness','valence']

radar_data = df.groupby('cluster_kmeans')[radar_features].mean()
# Normalize 0-1
radar_data = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

N = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

n_clusters_plot = len(radar_data)
fig, axes = plt.subplots(1, n_clusters_plot, figsize=(5 * n_clusters_plot, 5),
                         subplot_kw=dict(polar=True))
fig.suptitle('Audio Fingerprint per Cluster (Radar Charts)', fontsize=15, fontweight='bold')

if n_clusters_plot == 1:
    axes = [axes]

colors_radar = PALETTE[:n_clusters_plot]

for idx, (cluster_id, row) in enumerate(radar_data.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    ax = axes[idx]
    ax.set_facecolor('#1a1a2e')
    ax.fill(angles, values, color=colors_radar[idx], alpha=0.35)
    ax.plot(angles, values, color=colors_radar[idx], linewidth=2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_features, size=8, color='white')
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25', '0.5', '0.75'], size=6, color='#aaa')
    ax.set_title(f'C{cluster_id}\n{cluster_labels[cluster_id]}', size=10,
                 color='white', pad=15, fontweight='bold')
    ax.grid(color='#444', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('plot_14_radar_charts.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 16. Temporal Analysis — Cluster Evolution Across Decades

In [ ]:
decade_cluster = df.groupby(['decade', 'cluster_name']).size().unstack(fill_value=0)
decade_cluster_pct = decade_cluster.div(decade_cluster.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cluster Distribution Across Decades', fontsize=15, fontweight='bold')

# Absolute counts
decade_cluster.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='white', linewidth=0.3)
axes[0].set_title('Song Count per Cluster')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Number of Songs')
axes[0].legend(title='Cluster', fontsize=8)
axes[0].tick_params(axis='x', rotation=30)

# Percentage
decade_cluster_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10',
                        edgecolor='white', linewidth=0.3)
axes[1].set_title('Cluster Composition % per Decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Cluster', fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('plot_15_temporal_analysis.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(decade_cluster)

## 17. Supervised Validation — Feature Importance via Random Forest

In [ ]:
# Train RF to predict cluster labels — feature importance tells us what drives cluster separation
X_rf = df[AUDIO_FEATURES]
y_rf = df['cluster_kmeans']

X_tr, X_te, y_tr, y_te = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42, stratify=y_rf)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_tr, y_tr)

y_pred_rf = rf.predict(X_te)
print('Random Forest Cluster Prediction Report:')
print(classification_report(y_te, y_pred_rf))

importances = pd.Series(rf.feature_importances_, index=AUDIO_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importances.index, importances.values,
               color=[PALETTE[i % len(PALETTE)] for i in range(len(importances))])
ax.set_title('Feature Importance for Cluster Separation (Random Forest)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9, color='white')
plt.tight_layout()
plt.savefig('plot_16_feature_importance.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 18. Final Summary & Insights

In [ ]:
print('=' * 70)
print('           FINAL SUMMARY — SPOTIFY SONG COHORT DISCOVERY')
print('=' * 70)
print(f'Dataset          : Rolling Stones on Spotify')
print(f'Total tracks     : {len(df)}')
print(f'Audio features   : {len(AUDIO_FEATURES)}')
print(f'Decades spanned  : {df["decade"].nunique()} ({df["decade"].min()} – {df["decade"].max()})')
print()
print(f'K-Means (K={K_OPTIMAL}):  Silhouette = {sil_km:.4f}')
print(f'Agglomerative   :  Silhouette = {sil_agg2:.4f}')
print(f'DBSCAN          :  Clusters = {n_clusters_db}, Outliers = {n_noise}')
print(f'GMM (K={best_k_gmm})      :  Silhouette = {sil_gmm:.4f}')
print()
print('Discovered Song Cohorts:')
for k, v in cluster_labels.items():
    count = (df['cluster_kmeans'] == k).sum()
    avg_pop = df[df['cluster_kmeans'] == k]['popularity'].mean()
    print(f'  [{k}] {v:<30} | {count:>4} songs | Avg Popularity: {avg_pop:.1f}')
print()
print('Top 3 features driving cluster separation (by RF importance):')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {imp:.4f}')
print('=' * 70)